# 04 - GenAI

In [16]:
import json, time
import pandas as pd
from google import genai
from google.genai import types

MODEL_NAME = "gemini-3.1-flash-lite"

ALLOWED = ["forgot_or_no_reminder", "distance_or_transport", "cost_or_fee",
           "illness_or_emergency", "work_or_schedule", "other"]

SCHEMA = {
    "type": "object",
    "properties": {"root_cause": {"type": "string"}, "confidence": {"type": "number"},
                   "summary": {"type": "string"}, "recommended_action": {"type": "string"}},
    "required": ["root_cause", "confidence", "summary", "recommended_action"]}

API_KEY = input("Paste your key: ")         
client = genai.Client(api_key=API_KEY)

In [10]:
SYSTEM_TEXT = ("You are a patient-experience analyst for CareLoop, which runs outpatient clinics in India. "
               "You classify missed-appointment notes accurately and respectfully, and you reply in JSON only.")

def build_prompt(note_text):
    return f"""
Context: CareLoop runs outpatient clinics in India. When a patient misses an appointment, the CareLoop call centre phones them and writes a short note. Some notes are in mixed Hindi and English.
Role: Act as a Patient-Experience Analyst.
Action: Read the note and pick the one main reason from this list: {", ".join(ALLOWED)}. If the reason is unclear or not in the list, answer "other" rather than guessing. Give a confidence from 0 to 1, a summary of at most 25 words, and one recommended action for the clinic.
Tone: Neutral, factual and brief. Do not judge the patient.
Format: Return JSON only, with exactly 4 fields: root_cause, confidence, summary, recommended_action. No extra text before or after.

Note: {note_text}
"""

In [11]:
def call_gemini(note_text):
    """Sends one note; returns the RAW text reply (needed for 4.2's raw_response)."""
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=build_prompt(note_text),
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,                              
            max_output_tokens=1000,
            response_mime_type="application/json",
            response_schema=SCHEMA,
            thinking_config=types.ThinkingConfig(thinking_level="minimal")))
    return response.text

def analyse_note(note_text):
    return json.loads(call_gemini(note_text))           

notes = pd.read_csv("data/followup_notes.csv")

In [12]:
# 4.1 test: one API call
n001 = notes.loc[notes["note_id"] == "N001", "note_text"].iloc[0]
result = analyse_note(n001)
print(result)

{'root_cause': 'forgot_or_no_reminder', 'confidence': 1.0, 'summary': 'The patient missed the appointment because they did not receive a reminder and subsequently forgot the scheduled visit.', 'recommended_action': 'Verify patient contact details in the system and ensure SMS reminder settings are active for future appointments.'}


In [13]:
from pydantic import BaseModel, Field, ValidationError

class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)

FIELDS = ["root_cause", "confidence", "summary", "recommended_action"]
api_calls = 1                       # the N001 call from 4.1
rows = []

for _, n in notes.head(10).iterrows():                  
    text = n["note_text"]
    rec = {"note_id": n["note_id"], "appointment_id": n["appointment_id"],
           "root_cause": None, "confidence": None, "summary": None,
           "recommended_action": None, "validation_status": None, "raw_response": ""}

    # --- blank note: no API call ---
    if pd.isna(text) or str(text).strip() == "":
        rec.update(root_cause="other", confidence=0.0, summary="No note recorded",
                   recommended_action="Re-call patient", validation_status="skipped_blank")
    else:
        try:
            api_calls += 1
            raw = call_gemini(text)
            rec["raw_response"] = raw                          
            data = json.loads(raw)
            for k in FIELDS:                                   
                if k in data:
                    rec[k] = data[k]
            FollowupInsight(**data)                            
            if data["root_cause"] not in ALLOWED:
                raise ValueError("root_cause not in allowed list")
            rec["validation_status"] = "valid"
        except Exception as e:
            rec["validation_status"] = "invalid"
            print(n["note_id"], "invalid:", e)
        time.sleep(13)                                           
    rows.append(rec)
    print(n["note_id"], rec["validation_status"], rec["root_cause"])

out = pd.DataFrame(rows)
out.to_csv("notes_analysed.csv", index=False)
print(out.shape, "| API calls so far:", api_calls)

N001 valid forgot_or_no_reminder
N002 valid distance_or_transport
N003 valid cost_or_fee
N004 skipped_blank other
N005 valid illness_or_emergency
N006 valid work_or_schedule
N007 valid forgot_or_no_reminder
N008 valid distance_or_transport
N009 valid other
N010 valid work_or_schedule
(10, 8) | API calls so far: 10


In [14]:
# 4.3 (a) Feed Pydantic two bad replies -- no API call
bad_1 = {"root_cause": "cost_or_fee", "confidence": 1.7,
         "summary": "Patient found the fee too high.", "recommended_action": "Offer a discount."}
bad_2 = {"root_cause": "cost_or_fee", "confidence": 0.8,
         "summary": "Patient found the fee too high."}                     # recommended_action is missing

for name, reply in [("confidence = 1.7", bad_1), ("missing recommended_action", bad_2)]:
    try:
        FollowupInsight(**reply)
        print(name, "-> passed (unexpected!)")
    except ValidationError as e:
        print(name, "-> caught:\n", e, "\n")

confidence = 1.7 -> caught:
 1 validation error for FollowupInsight
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.7, input_type=float]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than_equal 

missing recommended_action -> caught:
 1 validation error for FollowupInsight
recommended_action
  Field required [type=missing, input_value={'root_cause': 'cost_or_f...ound the fee too high.'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing 



In [17]:
# 4.3 (b) Lazy prompt (V1) vs CRAFT prompt (V2) -- 4 API calls
def call_with_prompt(prompt_text):
    """Same settings as call_gemini, but takes a ready-made prompt."""
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt_text,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT, temperature=0, max_output_tokens=1000,
            response_mime_type="application/json", response_schema=SCHEMA,
            thinking_config=types.ThinkingConfig(thinking_level="minimal")))
    return json.loads(response.text)

results = []
for note_id in ["N007", "N009"]:
    text = notes.loc[notes["note_id"] == note_id, "note_text"].iloc[0]
    for version, prompt_text in [("V1", f"Classify this note: {text}"),
                                 ("V2", build_prompt(text))]:
        api_calls += 1
        reply = call_with_prompt(prompt_text)
        results.append({"note": note_id, "prompt": version,
                        "root_cause": reply.get("root_cause"),
                        "in_allowed_list": reply.get("root_cause") in ALLOWED})
        time.sleep(13)
print(pd.DataFrame(results))

   note prompt                               root_cause  in_allowed_list
0  N007     V1  Communication failure and forgetfulness            False
1  N007     V2                    forgot_or_no_reminder             True
2  N009     V1                    Communication failure            False
3  N009     V2                                    other             True


**Which prompt would I deploy?** V2, the CRAFT prompt. Its `root_cause` was in the allowed list (`in_allowed_list = True`) for both notes, while the lazy V1 prompt never sees the list, so it makes up its own labels (`in_allowed_list = False`), and those would fail validation and couldn't be counted or compared across clinics.

In [19]:
# 4.3 (c) Count the calls
print("Total API calls in Tasks 4.1–4.3:", api_calls)

Total API calls in Tasks 4.1–4.3: 15


In [20]:
# 4.4
clean = pd.read_csv("appointments_clean.csv")
merged = out.merge(clean[["appointment_id", "reminder_sent"]], on="appointment_id")   # BLANK 7
print(merged.shape)
print(pd.crosstab(merged["root_cause"], merged["reminder_sent"]))                    # BLANK 8

(10, 9)
reminder_sent          No  Yes
root_cause                    
cost_or_fee             0    1
distance_or_transport   1    1
forgot_or_no_reminder   2    0
illness_or_emergency    0    1
other                   1    1
work_or_schedule        1    1
